# GPU02 — خ۷ (F07) گروه ۷-ب: شبکه‌ی عصبی روی **سطح فرد (L5)** + تجمیع پواسون-دوجمله‌ای

> بند 7.16 (انتظار: 🟢 «جدی‌ترین شانس این خانواده») + بند 7.24.3 (تجمیع از سطح فرد).

**سؤال محوری این نوت‌بوک:** آیا یک مدل کامل سطح فرد — با embedding خودِ `PersonId`
روی ۲٬۰۴۹٬۳۲۲ رزرو و ۲۶٬۷۶۸ نفر — چیزی می‌دهد که فیچرهای تجمیع‌شده ندادند؟

⚠️ اسپرینت B همین فرضیه را از مسیر دیگری **رد کرد** (یافته‌ی ۲۱: فیچرهای کوهورت
L5→L1 کمک نکردند چون سیگنال فردی از قبل در فیچرهای rolling سلولی حاضر بود). این
نوت‌بوک همان فرضیه را در **قوی‌ترین شکل ممکنش** می‌آزماید. اگر این هم نبرد، رد
فرضیه دیگر «شاید تجمیع بد بود» ندارد — و این خودش یک نتیجه‌ی قابل‌گزارش است.

### نکته‌ی فنی محوری: تجمیع (بند 7.24.3)

مدل به‌ازای هر رزرو $p_i$ می‌دهد؛ تعداد عدم‌دریافت سلول = مجموع برنولی‌های
ناهم‌توزیع ⇒ **پواسون-دوجمله‌ای**، و کوانتایلش با بسط کورنیش-فیشر گرفته می‌شود.
ولی پواسون-دوجمله‌ای **استقلال افراد** را فرض می‌کند در حالی که F59 می‌گوید ۸۳٪
واریانس شوک مشترک روزانه است. به همین دلیل یک هایپرپارامتر صریح `overdispersion`
هست که Optuna تنظیمش می‌کند — **مقدار بهینه‌اش خودش اندازه‌گیری همبستگی درون‌روزی
است**، نه یک وصله.

**ارزیابی روی همان ردیف‌های L1** انجام می‌شود که همه‌ی خانواده‌های دیگر با آن سنجیده
شدند (بند 7.1.2)، وگرنه عدد pinball با هیچ ردیف دیگری قابل‌قیاس نبود.

**بودجه‌ی هدف: ~۱۰۰ دقیقه.** سنگین‌ترین نوت‌بوک از این چهارتا.

## سلول ۱ — نصب وابستگی‌ها

`torch`/`jax` روی کولب و کگل از پیش نصب‌اند و نسخه‌شان با درایور CUDA همان ماشین
هماهنگ است؛ نصب دوباره‌شان چند گیگابایت دانلود و گاهی ناسازگاری درایور می‌آورد.
پس فقط چیزهایی نصب می‌شوند که واقعاً نیستند. نسخه‌ی دقیق هرچه استفاده شد در سلول ۵
چاپ و در MLflow ثبت می‌شود (بازتولیدپذیری از راه **ثبت**، نه پین‌کردن).
فهرست کامل: `requirements-gpu.txt` داخل همین بسته.

In [1]:
!pip install -q optuna mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 14.1 MB/s eta 0:00:00


## سلول ۲ — بارگذاری بسته‌ی کد + داده

⚠️ **کد اصلی داخل نوت‌بوک نوشته نمی‌شود** (بند 7.8.4، قاعده‌ی «`notebooks/` = روایت،
`src/` = حقیقت»). این نوت‌بوک فقط `src/` را import و روایت می‌کند.

`gpu_bundle.zip` را با `python -m src.models.gpu_bundle` بسازید و در Drive بگذارید
(یا در کگل به‌عنوان Dataset آپلود کنید). داخلش: کل `src/`، چهار فایل
`data/processed/` که سلول ۳ رویشان assert می‌زند، و نتایج CPU خانواده‌های قبلی برای
جدول مقایسه.

In [2]:
MODE = "kaggle"          # ← "colab" یا "kaggle"
BUNDLE_COLAB  = "/content/drive/MyDrive/phase7/gpu_bundle.zip"   # ← مسیر خودتان
BUNDLE_KAGGLE = "/kaggle/working/t.zip"

import os, sys, zipfile, pathlib

if MODE == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    bundle, workdir = BUNDLE_COLAB, pathlib.Path("/content/phase7")
else:
    bundle, workdir = BUNDLE_KAGGLE, pathlib.Path("/kaggle/working/phase7")

workdir.mkdir(parents=True, exist_ok=True)
# with zipfile.ZipFile(bundle) as z:
#     z.extractall(workdir)
os.chdir(workdir)
sys.path.insert(0, str(workdir))
print("محتوای بسته:", sorted(p.name for p in workdir.iterdir()))

محتوای بسته: []


In [3]:
import os
import shutil

src_dir = "/kaggle/input/datasets/mvajhi/bundle"
dest_dir = "/kaggle/working/phase7"

os.makedirs(dest_dir, exist_ok=True)

# Copy all contents from Kaggle input to working/my_dir
for item in os.listdir(src_dir):
    src_path = os.path.join(src_dir, item)
    dest_path = os.path.join(dest_dir, item)
    
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dest_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dest_path)
os.chdir(workdir)
sys.path.insert(0, str(workdir))
print("محتوای بسته:", sorted(p.name for p in workdir.iterdir()))

محتوای بسته: ['AGENTS.md', 'BUNDLE_INFO.json', 'data', 'doc', 'reports', 'requirements-gpu.txt', 'src']


## سلول ۳ — ⭐ دروازه‌ی انصاف A1 (بند 7.7.3)

**اگر هش‌ها نخوانند، نوت‌بوک همین‌جا می‌ایستد.** بدون این assert هیچ اثباتی وجود
ندارد که این اجرا روی همان foldها و همان snapshot دادهٔ خانواده‌های CPU انجام شده —
و هر run با `cv_folds_hash` نامنطبق از جدول مقایسه‌ی فاز ۷ حذف می‌شود. مقادیر زیر
از بخش «قفل فاز ۷» `doc/data_manifest.md` آمده‌اند.

In [4]:
from src.models.gpu_runner import assert_fairness_gate

EXPECTED_CV_FOLDS_HASH      = "bd08d6f7c801ee0611121e404774251de07a480ac1589b12eb7c64f8044b78d4"
EXPECTED_DATA_SNAPSHOT_HASH = "90097e5f3b7d4572ee94c9a1a09ae0d0ec65115c413445a4751e20c48ac35de0"   # data/processed/person_features_v1.parquet

from src.models.families.f07_neural_l5 import load_l5_bridge
data = load_l5_bridge()   # ارزیابی L1، آموزش L5 — بند بالای ماژول

assert_fairness_gate(data, EXPECTED_CV_FOLDS_HASH, EXPECTED_DATA_SNAPSHOT_HASH)
print(data.summary())

✅ دروازه‌ی انصاف A1 پاس شد · cv_folds_hash=bd08d6f7c801… · data_snapshot_hash=90097e5f3b7d…
L5 — 7,579 ردیف، 5 fold (fold0: 3,556→870 · fold1: 4,426→860 · fold2: 5,286→185 · fold3: 5,471→1,025 · fold4: 6,496→940)


## سلول ۴ — بذر تصادفی سراسری

`set_global_seed()` تنها منبع بذر پروژه است (`AGENTS.md`). قطعیت کامل روی GPU
تضمین‌شدنی نیست — به‌همین‌دلیل قاعده‌ی **سه seed** (A7، بند 7.16.3) در مرحله‌ی
قهرمان اجرا می‌شود و پراکندگی بین seedها خودش گزارش می‌گردد، نه پنهان.

In [5]:
from src.config import set_global_seed
from src.models.gpu_runner import setup_torch_determinism

set_global_seed()
setup_torch_determinism(strict=False)   # strict=True بعضی op های cuDNN را می‌شکند

## سلول ۵ — سخت‌افزار

زمان‌های اجرا فقط با دانستن سخت‌افزار قابل تفسیرند (بند 7.8.2).

In [6]:
!nvidia-smi

from src.models.gpu_runner import device_report

DEVICE = device_report()
DEVICE

Sun Aug 16 16:13:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

INFO:2026-08-16 16:13:05,949:jax._src.xla_bridge:822: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2026-08-16 16:13:05,949 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


{'platform': 'Linux-6.12.90+-x86_64-with-glibc2.35',
 'python': '3.12.13',
 'torch': '2.10.0+cu128',
 'cuda': '12.8',
 'device': 'cuda',
 'gpu_name': 'Tesla T4',
 'gpu_memory_gb': 15.64,
 'jax': '0.7.2',
 'jax_devices': ['cuda:0', 'cuda:1']}

## سلول ۶ — ردیابی MLflow جدا

`mlruns_gpu/` جداست تا ادغام با `mlruns/` محلی (بند 7.8.3 گام ۵) امن و قابل بازگشت
باشد. tag اجباری `compute` هم همین‌جا ست می‌شود.

In [7]:
from src.models.gpu_runner import use_gpu_tracking

COMPUTE = "kaggle"     # اگر روی کگل اجرا می‌کنید: "kaggle"
print("MLflow →", use_gpu_tracking("mlruns_gpu"))

MLflow → /kaggle/working/phase7/mlruns_gpu


## سلول ۷-الف — R0: آزمایش دود روی ۲ میلیون رزرو

با هایپرپارامتر پیش‌فرض و epoch کم — فقط برای اثبات اینکه پایپ‌لاین کامل
(برازش سطح فرد ← احتمال هر رزرو ← تجمیع پواسون-دوجمله‌ای ← merge روی سلول‌های L1)
سرتاسر کار می‌کند و عدد معنادار می‌دهد.

In [8]:
from src.models.families import f07_neural_l5 as fam
from src.models.gpu_runner import smoke_test

person = fam.load_person_frame()
print(f"جدول رزرو فردی: {len(person):,} ردیف · {person['PersonId'].nunique():,} فرد · "
      f"نرخ عدم‌دریافت کل = {person['dont_receive'].mean():.4f}")

smoke = [smoke_test(fam.FITTERS["mlp_embedding_l5"], data,
                    hyperparams={"epochs": 3, "patience": 2})]

جدول رزرو فردی: 2,049,322 ردیف · 26,768 فرد · نرخ عدم‌دریافت کل = 0.0807
R0 mlp_embedding_l5             pinball=0.01468 (B3=0.01375) پوشش=0.126 R²=-0.402 13.4s


## سلول ۷-ب — R2: تنظیم روی ۳ fold نخست

هر برازش این‌جا گران است (میلیون‌ها ردیف)، پس منطق R1 بند 7.3.2 اعمال می‌شود:
**تنظیم روی ۳ fold نخست، تأیید نهایی روی هر ۵**. این دقیقاً همان صرفه‌جویی است که
سند تصمیم ۳۷ خواسته — بدون آن، هر trial پنج برازش کامل می‌شد.

In [9]:
from src.models.gpu_runner import run_gpu_study
from src.models.spaces import SPACES

study = run_gpu_study(
    fam.FITTERS["mlp_embedding_l5"], SPACES["mlp_embedding_l5"].fn,
    data.first_folds(3),                       # ← غربالگری ارزان، بند 7.3.2
    family=fam.FAMILY, feature_set=fam.FEATURE_SET,
    budget_minutes=20, compute=COMPUTE, seed=42,
    target="binary", output_aggregation="aggregated")   # ← محورها صریح، بند 7.9.1

2026/08/16 16:13:22 INFO mlflow.tracking.fluent: Experiment with name 'phase7' does not exist. Creating a new experiment.



R2 — F07/mlp_embedding_l5 (L5) · τ=0.2 · بودجه=20 دقیقه · دستگاه=cuda · مرجع B3=0.01954


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

  trial   0 | pinball=0.01960 | بهترین=0.01960 |  39.5s | گذشته=  0.7/20 دقیقه


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

...

  trial  43 | pinball=0.01904 🎯 | بهترین=0.01874 |  20.3s | گذشته= 19.4/20 دقیقه


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

  trial  44 | pinball=0.01929 🎯 | بهترین=0.01874 |  20.8s | گذشته= 19.8/20 دقیقه


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

  trial  45 | pinball=0.01858 🎯 | بهترین=0.01858 |  20.6s | گذشته= 20.1/20 دقیقه
⏱️  بودجه‌ی زمانی (20 دقیقه) تمام شد پس از 46 trial — با بهترین نتیجه‌ی تا این لحظه ادامه می‌دهیم.

✅ mlp_embedding_l5: بهترین pinball=0.01858 در برابر B3=0.01954 (برد) · 46 trial · همگرا(A6)=⚠️ · پایداری=2/3


## سلول ۷-ج — قهرمان روی هر ۵ fold + ACI + DM

دو seed (نه سه): هر seed این‌جا یعنی ۵ برازش کامل روی میلیون‌ها ردیف. پراکندگی
بین همین دو هم اگر بزرگ باشد، کافی است که «برد» را زیر سؤال ببرد.

In [10]:
from src.models.gpu_runner import finalize_champion

champion = finalize_champion(fam.FITTERS["mlp_embedding_l5"], data, study,
                             feature_set=fam.FEATURE_SET, seeds=(42, 1234),
                             compute=COMPUTE, run_aci=True,
                             target="binary", output_aggregation="aggregated")
champions = [champion]

  seed 42: pinball(ردیفی)=0.01762
  seed 1234: pinball(ردیفی)=0.01800
  ACI: پوشش=0.2106 (شکاف +0.0106) · pinball=0.01404

🏁 mlp_embedding_l5: pinball(ردیفی)=0.01762 در برابر B3=0.01335 · DM p=0.0000 ❌ غیرمعنادار · 20 فایل مدل ذخیره شد


/usr/local/lib/python3.12/dist-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

## سلول ۷-د — ⭐ آزمون تجمیع (بند 7.24.3): کوانتایل پواسون-دوجمله‌ای در برابر میانگین

سه حالت روی همان مدل و همان احتمال‌ها مقایسه می‌شوند:

| حالت | معنی |
|---|---|
| `mean_only` | فقط میانگین $\sum p_i / n$ — یعنی هیچ کوانتایلی گرفته نشده |
| `cornish_fisher`, `overdispersion=1` | کوانتایل با فرض **استقلال کامل** افراد |
| `cornish_fisher`, `overdispersion` تنظیم‌شده | با احتساب همبستگی درون‌روزی |

فاصله‌ی ردیف دوم تا سوم، **قیمت فرض استقلال** است — همان چیزی که بند 7.24.3 هشدار
می‌دهد و تا امروز در این پروژه هرگز کمّی نشده بود.

In [11]:
import numpy as np, pandas as pd
from src.baselines import operational_metrics
from src.models.axes import TUNING_TAU

hp = dict(study.best_hyperparams)
tr0, te0 = data.folds[0]
model0 = fam.FITTERS["mlp_embedding_l5"].fit(tr0, TUNING_TAU, **hp)

rows = []
for label, mode, od in [("میانگین (بدون کوانتایل)", "mean_only", 1.0),
                        ("پواسون-دوجمله‌ای، استقلال کامل", "cornish_fisher", 1.0),
                        (f"پواسون-دوجمله‌ای، overdispersion={hp.get('overdispersion', 1.0):.2f}",
                         "cornish_fisher", float(hp.get("overdispersion", 1.0)))]:
    model0.arch["aggregation_mode"], model0.arch["overdispersion"] = mode, od
    pred = fam.predict_l5(model0, te0, TUNING_TAU)
    m = operational_metrics(te0, pred, TUNING_TAU)
    rows.append({"حالت تجمیع": label, "pinball": round(m["pinball"], 5),
                 "پوشش": round(m["coverage"], 4), "شکاف از τ": round(m["coverage_gap"], 4),
                 "نرخ کمبود": round(m["shortage_rate"], 4)})

aggregation_table = pd.DataFrame(rows)
aggregation_table.to_csv("reports/gpu/F07b_aggregation_ablation.csv", index=False)
aggregation_table

,حالت تجمیع,pinball,پوشش,شکاف از τ,نرخ کمبود
0,میانگین (بدون کوانتایل),0.01611,0.3414,0.1414,0.2172
1,پواسون-دوجمله‌ای، استقلال کامل,0.01413,0.2310,0.0310,0.1253
2,پواسون-دوجمله‌ای، overdispersion=1.31,0.01396,0.2000,0.0000,0.1103


## سلول ۷-ه — راستی‌آزمایی مدل ذخیره‌شده

In [12]:
from pathlib import Path

stem = Path("models/gpu/F07/mlp_embedding_l5/mlp_embedding_l5__s42__fold0")
reloaded = fam.FITTERS["mlp_embedding_l5"].load(stem)
pred_reloaded = fam.predict_l5(reloaded, data.folds[0][1], TUNING_TAU)
print(f"پارامترها: {reloaded.n_parameters:,} · ردیف‌های آموزش سطح فرد: "
      f"{reloaded.arch['n_person_rows_train']:,}")
assert np.isfinite(pred_reloaded).all()
print("✅ مدل ذخیره‌شده قابل استفاده است")

پارامترها: 1,152,016 · ردیف‌های آموزش سطح فرد: 1,021,036
✅ مدل ذخیره‌شده قابل استفاده است


## سلول ۷-و — گزارش فارسی کامل

In [13]:
from src.models.gpu_runner import render_family_report, save_family_report

notes = [
    "مدل روی L5 آموزش دید ولی ارزیابی روی همان ردیف‌های L1 انجام شد (بند 7.1.2) — "
    "چون داده‌ی فردی FoodType ندارد، پیش‌بینی هر سلول (d,m,r) روی همه‌ی سطرهای هم‌غذا پخش شد.",
    f"overdispersion بهینه = {study.best_hyperparams.get('overdispersion', float('nan')):.3f} — "
    "بزرگ‌تر از ۱ یعنی فرض استقلال افراد (بند 7.24.3) واقعاً نقض می‌شود، سازگار با F59.",
    "این نوت‌بوک فرضیه‌ی یافته‌ی ۲۱ (سیگنال فردی افزونه است) را در قوی‌ترین شکلش می‌آزماید.",
    "تقسیم fold بر اساس تاریخ است نه فرد (بند 7.16.3) — همان فرد می‌تواند در train و test باشد.",
    "⚠️ ستون B3 جدول R2 روی ۳ fold نخست است (تنظیم آن‌جا انجام شد) ولی جدول قهرمان روی هر ۵ fold — "
    "این دو عدد مستقیماً قابل‌قیاس نیستند و عمداً جدا گزارش شده‌اند.",
]
report = render_family_report(
    "F07", "خ۷ گروه ۷-ب — شبکه‌ی عصبی سطح فرد (L5) با تجمیع پواسون-دوجمله‌ای",
    [study], champions, smoke, DEVICE, notes)
save_family_report("F07", report, "F07b_neural_L5")
print(report)

گزارش نوشته شد: /kaggle/working/phase7/reports/gpu/F07b_neural_L5.md
# خ۷ گروه ۷-ب — شبکه‌ی عصبی سطح فرد (L5) با تجمیع پواسون-دوجمله‌ای

> اجرای GPU، بند 7.8 `doc/WBS-phase7-modeling.md`. خانواده F07. سخت‌افزار: Tesla T4 · torch=2.10.0+cu128 · CUDA=12.8

## R0 — آزمایش دود (اجراپذیری + سیم‌چین نشتی)

| مدل | pinball | B3 | پوشش | R² | زمان |
|---|---|---|---|---|---|
| `mlp_embedding_l5` | 0.01468 | 0.01375 | 0.126 | -0.402 | 13.4s |

## R2 — تنظیم با بودجه‌ی زمانی

| مدل | بهترین pinball | B3 | trial | همگرا (A6) | پایداری (۷.۶.۳) | شکست | ساعت-هسته |
|---|---|---|---|---|---|---|---|
| `mlp_embedding_l5` 🎯 | **0.01858** | 0.01954 | 46 | ⚠️ | 2/5 | 0 | 0.34h |

## S3 — قهرمان‌ها: سه seed، کالیبراسیون ACI، آزمون Diebold-Mariano

| مدل | pinball(ردیفی) | B3(ردیفی) | Δ | DM p | معنادار؟ | پوشش | پوشش پس از ACI |
|---|---|---|---|---|---|---|---|
| `mlp_embedding_l5` | 0.01762 | 0.01335 | +0.00427 | 0.0000 | ❌ خیر | 0.4041 (+0.2041) | 0.2106 (+0.0106) |

### پراکندگی بین seedها (قاعده‌ی A

## سلول ۸ — بسته‌بندی خروجی (تکه‌های ۱۰۰ مگابایتی)

همه‌ی خروجی‌ها — `mlruns_gpu/` (هر trial + قهرمان‌ها با artifact مدل)،
`models/gpu/` (وزن‌ها و پیش‌پردازش هر fold/seed)، `reports/gpu/` (JSON و گزارش
فارسی)، و `optuna_studies/*.db` (تا اجرای بعدی از همین‌جا ادامه دهد) — در یک zip
جمع و به تکه‌های ۱۰۰ مگابایتی شکسته می‌شوند. هر تکه SHA-256 خودش را در
`MANIFEST_F07b_neural_L5.json` دارد، پس اگر دانلود یکی خراب شد فقط همان یکی دوباره گرفته
می‌شود.

In [14]:
from src.models.gpu_runner import package_outputs, download_parts

manifest = package_outputs(tag="F07b_neural_L5", part_mb=100)
download_parts()      # روی کولب دانلود می‌کند؛ روی کگل فایل‌ها در خروجی session می‌مانند


📦 بسته‌بندی شد: 3,417 فایل (104.4 MB خام) → 1 تکه در gpu_outputs/
   gpu_outputs_F07b_neural_L5.zip  91.3 MB

بازیابی محلی:
   unzip gpu_outputs_F07b_neural_L5.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## پس از اجرا — چرخه‌ی بازگشت (بند 7.8.3)

```
۱. این نوت‌بوک اجراشده (File → Download .ipynb، با تمام خروجی‌ها) →
   notebooks/gpu/executed/{name}__{تاریخ}.ipynb    ← حتی اگر آزمایش شکست خورد؛ شکست هم داده است
۲. تکه‌ها → ریشه‌ی مخزن:  cat gpu_outputs_*.zip.part* > gpu_outputs.zip && unzip gpu_outputs.zip
۳. rsync -a mlruns_gpu/ mlruns/        (ادغام MLflow)
۴. کارت مدل ۱۴ گامی → reports/models/{model_id}.md
۵. make mlflow-ui  →  runهای جدید با tag compute=colab باید دیده شوند
```